In [ ]:
# 开箱即用，无需自己加载模型。但在实际的生产环境或需要特定中文效果的项目中，我们通常需要显式地指定嵌入模型
# pip install chromadb
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import numpy as np

In [ ]:
# 1. 初始数据库，创建客户端：PersistentClient使数据持久化到本地磁盘
client = chromadb.PersistentClient(path="./chroma_db")

# 2. 创建 / 获取集合
collection = client.get_or_create_collection(   # get_or_create_collection：如果集合存在则获取，不存在则创建
    name="document_collection",
    metadata={"hnsw:space": "cosine"}           # 向量计算方式：余弦相似度
)

# 3. 准备数据
documents = [                                   # 文档内容
    "向量数据库是AI应用的核心基础设施",
    "RAG系统需要高效的向量检索能力",
    "相似度搜索是向量数据库的核心功能"
]

ids = ["doc1", "doc2", "doc3"]                   # 每个文档的唯一标识

metadatas = [                                    # 每个文档的元数据（进一步过滤）
    {"category": "技术", "source": "文档1"},
    {"category": "技术", "source": "文档2"},
    {"category": "技术", "source": "文档3"}
]

# 4. 添加文档
# Chroma会自动把 documents 转换成 embedding
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

# 5. 查询
# Chroma会自动把 query_text 转换成 embedding
results = collection.query(
    query_texts=["向量数据库的核心功能"],
    n_results=2,                                # 返回 Top-K
    where={"category": "技术"}                  # 元数据过滤
)

# 6. 查看结果
print("Chroma查询结果:")

for i, doc in enumerate(results["documents"][0]):
    print(f"{i + 1}. {doc}")
    print(f"  距离：{results['distances'][0][i]:.4f}")